# SIC8 | Historical Transactions — Source Profiling

**Purpose:** Inspect the source before cleaning. Read-only; no Delta/S3 writes.

Run `SAMPLE` first, then `FULL`. Source fraud labels are for profiling—not detection input.

## 01 — Settings
Choose scope and point to the transactions CSV (not the folder).

In [0]:
# Project settings
RAW_PATH = "/Volumes/workspace/sic8_sources/historical_raw"
TRANSACTIONS_PATH = f"{RAW_PATH}/credit_card_transactions-ibm_v2.csv"
PROFILE_MODE = "SAMPLE"     # "SAMPLE" or "FULL"
SAMPLE_ROWS = 5000

# Prior report (verify in FULL mode)
PREVIOUS_TRANSACTION_COUNT = 24_386_900
PREVIOUS_FRAUD_COUNT = 29_757

EXPECTED_COLUMNS = [
    "User", "Card", "Year", "Month", "Day", "Time", "Amount",
    "Use Chip", "Merchant Name", "Merchant City", "Merchant State",
    "Zip", "MCC", "Errors?", "Is Fraud?"
]

print("Mode:", PROFILE_MODE)
print("Source:", TRANSACTIONS_PATH)


Mode: SAMPLE
Source: /Volumes/workspace/sic8_sources/historical_raw/credit_card_transactions-ibm_v2.csv


## 02 — Environment
Check the Spark version, catalog and source files. Compute details are in the notebook's Compute menu.

In [0]:
# Spark / catalog / source inventory
print("Spark version:", spark.version)
spark.sql("SELECT current_catalog() AS catalog, current_schema() AS schema").show(truncate=False)

for item in dbutils.fs.ls(RAW_PATH):
    print(item.name, "| bytes:", item.size)

print("Compute type: check the notebook Compute selector.")


Spark version: 4.2.0
+---------+-------+
|catalog  |schema |
+---------+-------+
|workspace|default|
+---------+-------+

city_reference.csv | bytes: 104813
credit_card_transactions-ibm_v2.csv | bytes: 2350744057
merchant_category_codes.csv | bytes: 52651
sd254_cards.csv | bytes: 487120
sd254_users.csv | bytes: 224394
Compute type: check the notebook Compute selector.


## 03 — Source read
Load the 15-column transaction CSV. Keep raw values as strings; apply explicit types later.

In [0]:
# Read only the transaction file
from pyspark.sql import functions as F
import time

start = time.perf_counter()
transactions_raw = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "false")
    .option("enforceSchema", "false")
    .option("mode", "PERMISSIVE")
    .csv(TRANSACTIONS_PATH)
)

print("DataFrame setup:", round(time.perf_counter() - start, 2), "sec (lazy)")
print("Columns:", len(transactions_raw.columns))
if transactions_raw.columns != EXPECTED_COLUMNS:
    raise ValueError(f"Unexpected source columns: {transactions_raw.columns}")
transactions_raw.printSchema()


DataFrame setup: 0.0 sec (lazy)
Columns: 15
root
 |-- User: string (nullable = true)
 |-- Card: string (nullable = true)
 |-- Year: string (nullable = true)
 |-- Month: string (nullable = true)
 |-- Day: string (nullable = true)
 |-- Time: string (nullable = true)
 |-- Amount: string (nullable = true)
 |-- Use Chip: string (nullable = true)
 |-- Merchant Name: string (nullable = true)
 |-- Merchant City: string (nullable = true)
 |-- Merchant State: string (nullable = true)
 |-- Zip: string (nullable = true)
 |-- MCC: string (nullable = true)
 |-- Errors?: string (nullable = true)
 |-- Is Fraud?: string (nullable = true)



## 04 — Connection timing
Compare two tiny reads. These numbers do **not** measure full-dataset throughput.

In [0]:
# Two small reads; not a full-scan benchmark
preview_columns = ["User", "Card", "Year", "Month", "Day", "Time",
                   "Amount", "Use Chip", "MCC", "Is Fraud?"]

start = time.perf_counter()
display(transactions_raw.select(*preview_columns).limit(5))
print("First preview:", round(time.perf_counter() - start, 2), "sec")

start = time.perf_counter()
second_preview = transactions_raw.select("User", "Card", "Amount").limit(5).collect()
print("Second preview:", round(time.perf_counter() - start, 2), "sec")
print("Rows returned:", len(second_preview))


User,Card,Year,Month,Day,Time,Amount,Use Chip,MCC,Is Fraud?
0,0,2002,9,1,06:21,$134.09,Swipe Transaction,5300,No
0,0,2002,9,1,06:42,$38.48,Swipe Transaction,5411,No
0,0,2002,9,2,06:22,$120.34,Swipe Transaction,5411,No
0,0,2002,9,2,17:45,$128.95,Swipe Transaction,5651,No
0,0,2002,9,3,06:23,$104.71,Swipe Transaction,5912,No


First preview: 1.59 sec
Second preview: 0.72 sec
Rows returned: 5


## 05 — Scope
`SAMPLE` checks up to 5,000 rows. It is neither random nor representative. `FULL` checks all rows.

In [0]:
# Select profiling scope
if PROFILE_MODE not in ("SAMPLE", "FULL"):
    raise ValueError("PROFILE_MODE must be SAMPLE or FULL")

working_df = (transactions_raw.limit(SAMPLE_ROWS)
              if PROFILE_MODE == "SAMPLE" else transactions_raw)
print("Profiling mode:", PROFILE_MODE)
print("SAMPLE is not random and may vary across executions.")


Profiling mode: SAMPLE
SAMPLE is not random and may vary across executions.


## 06 — Parse checks
Create temporary parsed amount and timestamp columns to detect errors. Preserve the original fields.

In [0]:
# Temporary parsed fields; raw fields remain unchanged
profile_df = (
    working_df
    .withColumn(
        "_amount_decimal",
        F.expr("try_cast(regexp_replace(trim(`Amount`), '[$,]', '') AS DECIMAL(18,2))")
    )
    .withColumn(
        "_parsed_event_time",
        F.expr("""
            try_to_timestamp(concat(
                lpad(trim(`Year`), 4, '0'), '-',
                lpad(trim(`Month`), 2, '0'), '-',
                lpad(trim(`Day`), 2, '0'), ' ', trim(`Time`)
            ), 'yyyy-MM-dd HH:mm')
        """)
    )
)
print("Temporary columns added; no data written.")


Temporary columns added; no data written.


## 07 — Quality metrics
Count missing and invalid values with one Spark aggregation. This is not a complete CSV-structure validator.

In [0]:
# Aggregate missingness, parsing and category counts in one action
from pyspark.sql.functions import col

def is_missing(name):
    return col(name).isNull() | (F.trim(col(name)) == "")

def count_if(condition, name):
    return F.sum(F.when(condition, 1).otherwise(0)).cast("long").alias(name)

metrics = [F.count("*").alias("total_rows")]
for name in EXPECTED_COLUMNS:
    metrics.append(count_if(is_missing(name), f"missing__{name}"))

metrics += [
    count_if(~is_missing("Amount") & col("_amount_decimal").isNull(), "invalid_amount"),
    count_if(col("_amount_decimal") < 0, "negative_amount"),
    count_if(col("_amount_decimal") == 0, "zero_amount"),
    count_if(~is_missing("Time") & col("_parsed_event_time").isNull(), "invalid_event_time"),
    count_if(~is_missing("User") & ~F.trim(col("User")).rlike(r"^[0-9]+$"), "invalid_user_id"),
    count_if(~is_missing("Card") & ~F.trim(col("Card")).rlike(r"^[0-9]+$"), "invalid_card_index"),
    count_if(~is_missing("MCC") & ~F.trim(col("MCC")).rlike(r"^[0-9]{4}$"), "invalid_mcc_format"),
    count_if(~is_missing("Is Fraud?") & ~F.trim(col("Is Fraud?")).isin("Yes", "No"), "unknown_fraud_label"),
    count_if(col("Is Fraud?") == "Yes", "fraud_yes"),
    count_if(col("Is Fraud?") == "No", "fraud_no"),
    count_if(col("Use Chip") == "Swipe Transaction", "method_swipe"),
    count_if(col("Use Chip") == "Chip Transaction", "method_chip"),
    count_if(col("Use Chip") == "Online Transaction", "method_online"),
    F.min("_amount_decimal").alias("min_amount"),
    F.max("_amount_decimal").alias("max_amount"),
    F.min("_parsed_event_time").alias("min_event_time_local_unknown_tz"),
    F.max("_parsed_event_time").alias("max_event_time_local_unknown_tz"),
]

start = time.perf_counter()
profile_result = profile_df.agg(*metrics).first().asDict()  # Spark action
profile_seconds = round(time.perf_counter() - start, 2)

print("Mode:", PROFILE_MODE, "| Rows:", f"{profile_result['total_rows']:,}",
      "| Duration:", profile_seconds, "sec")
display(spark.createDataFrame(
    [(key, str(value)) for key, value in profile_result.items()],
    ["metric", "value"]
))


Mode: SAMPLE | Rows: 5,000 | Duration: 4.63 sec


metric,value
total_rows,5000
missing__User,0
missing__Card,0
missing__Year,0
missing__Month,0
missing__Day,0
missing__Time,0
missing__Amount,0
missing__Use Chip,0
missing__Merchant Name,0


## 08 — Distributions
Show fraud labels and payment methods using the aggregate result (no extra source scan).

In [0]:
# Reuse counts from the same scan (no second groupBy scan)
fraud_counts = [("No", profile_result["fraud_no"]),
                ("Yes", profile_result["fraud_yes"])]
method_counts = [("Swipe Transaction", profile_result["method_swipe"]),
                 ("Chip Transaction", profile_result["method_chip"]),
                 ("Online Transaction", profile_result["method_online"])]

print("Fraud labels (source ground truth):")
display(spark.createDataFrame(fraud_counts, ["Is Fraud?", "count"]))
print("Payment methods:")
display(spark.createDataFrame(method_counts, ["Use Chip", "count"]))

print("Rows covered by known labels:", sum(x[1] for x in fraud_counts))
print("Rows covered by known methods:", sum(x[1] for x in method_counts))


Fraud labels (source ground truth):


Is Fraud?,count
No,4987
Yes,13


Payment methods:


Use Chip,count
Swipe Transaction,3732
Chip Transaction,975
Online Transaction,293


Rows covered by known labels: 5000
Rows covered by known methods: 5000


## 09 — Review flags
Quick checks only. Investigate warnings before designing Silver transformations.

In [0]:
# Flags are for review, not automatic row deletion
checks = {
    "Required IDs present": profile_result["missing__User"] == 0 and profile_result["missing__Card"] == 0,
    "Amounts parse": profile_result["invalid_amount"] == 0 and profile_result["missing__Amount"] == 0,
    "Event timestamps parse": profile_result["invalid_event_time"] == 0,
    "Fraud labels recognized": profile_result["unknown_fraud_label"] == 0,
}
for name, passed in checks.items():
    print(("OK" if passed else "REVIEW"), "-", name)
print("Missing location/errors and negative amounts need interpretation, not automatic filling.")
print("SAMPLE results are not full-source validation.")


OK - Required IDs present
OK - Amounts parse
OK - Event timestamps parse
OK - Fraud labels recognized
Missing location/errors and negative amounts need interpretation, not automatic filling.
SAMPLE results are not full-source validation.


In [0]:
# Share this summary after SAMPLE or FULL
rows = profile_result["total_rows"]
print("SIC8 historical source profiling")
print("Mode:", PROFILE_MODE, "| File:", TRANSACTIONS_PATH)
print("Rows:", f"{rows:,}", "| Columns:", len(transactions_raw.columns))
for name in ["missing__Merchant State", "missing__Zip", "missing__Errors?",
             "invalid_amount", "negative_amount", "invalid_event_time", "fraud_yes"]:
    print(name, ":", profile_result[name])
print("Main aggregation time:", profile_seconds, "sec")
if PROFILE_MODE == "FULL":
    print("Prior row-count difference:", rows - PREVIOUS_TRANSACTION_COUNT)
    print("Prior fraud-count difference:", profile_result["fraud_yes"] - PREVIOUS_FRAUD_COUNT)
else:
    print("SAMPLE only: do not use this to estimate dataset-wide rates.")
print("No cleaning or Delta writes performed.")


SIC8 historical source profiling
Mode: SAMPLE | File: /Volumes/workspace/sic8_sources/historical_raw/credit_card_transactions-ibm_v2.csv
Rows: 5,000 | Columns: 15
missing__Merchant State : 295
missing__Zip : 367
missing__Errors? : 4874
invalid_amount : 0
negative_amount : 174
invalid_event_time : 0
fraud_yes : 13
Main aggregation time: 4.63 sec
SAMPLE only: do not use this to estimate dataset-wide rates.
No cleaning or Delta writes performed.


## 11 — Next step
Review the output, then implement **canonical mapping** in a separate notebook: types, deterministic IDs, source lineage, and quality gates.

For performance, check **See performance** on Serverless; Classic clusters expose a Spark UI.